In [2]:
# ==========================================================
# Wine Quality - AutoGluon (STREAMLIT READY VERSION)
# ==========================================================
# ✅ Pasta fixa (autogluon_model) -> Streamlit único funciona
# ✅ EDA + gráfico
# ✅ Feature engineering
# ✅ meta.json padronizado
# ==========================================================

import os
import json
import glob
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

from ydata_profiling import ProfileReport
import kagglehub

from autogluon.tabular import TabularPredictor

# =========================
# CONFIGURAÇÕES
# =========================
RANDOM_STATE = 42
TEST_SIZE = 0.2
TIME_LIMIT = 180  # segundos

BASE_DIR = os.path.abspath("wine_autogluon")
MODEL_DIR = os.path.join(BASE_DIR, "autogluon_model")  # ✅ pasta fixa (streamlit)

DIR_REPORTS = os.path.join(BASE_DIR, "reports")
DIR_FIGURES = os.path.join(BASE_DIR, "figures")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DIR_REPORTS, exist_ok=True)
os.makedirs(DIR_FIGURES, exist_ok=True)

print("✅ AutoGluon salvo em:", MODEL_DIR)

# =========================
# FUNÇÕES AUXILIARES
# =========================
def load_dataset():
    path = kagglehub.dataset_download("yasserh/wine-quality-dataset")
    csv = [c for c in glob.glob(os.path.join(path, "*.csv")) if "WineQT" in c][0]
    df = pd.read_csv(csv).drop(columns=["Id"])
    return df


def normalize_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df


def feature_engineering(df):
    eps = 1e-9
    df = df.copy()
    df["total_acidity"] = df["fixed_acidity"] + df["volatile_acidity"] + df["citric_acid"]
    df["alcohol_sugar_ratio"] = df["alcohol"] / (df["residual_sugar"] + eps)
    df["so2_ratio"] = df["free_sulfur_dioxide"] / (df["total_sulfur_dioxide"] + eps)
    df["density_alcohol"] = df["density"] * df["alcohol"]
    return df


def plot_balance(y):
    plt.figure(figsize=(6,4))
    sns.countplot(x=y, color="maroon")
    plt.title("Distribuição das Classes")
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_FIGURES, "class_balance.png"), dpi=200)
    plt.close()

# =========================
# MAIN
# =========================
def main():
    print("[1/5] Carregando dados...")
    df = load_dataset()
    df = normalize_columns(df)

    print("[2/5] EDA...")
    ProfileReport(df, title="Wine Quality - EDA (AutoGluon)", explorative=True)\
        .to_file(os.path.join(DIR_REPORTS, "eda_wine_quality.html"))

    plot_balance(df["quality"])

    print("[3/5] Feature Engineering...")
    df = feature_engineering(df)

    train_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        stratify=df["quality"],
        random_state=RANDOM_STATE
    )

    print("[4/5] Treinando AutoGluon...")
    predictor = TabularPredictor(
        label="quality",
        problem_type="multiclass",
        eval_metric="f1_macro",
        path=MODEL_DIR
    )

    predictor.fit(
        train_data=train_df,
        time_limit=TIME_LIMIT,
        presets="best_quality",
        verbosity=2
    )

    y_pred = predictor.predict(test_df)
    f1 = f1_score(test_df["quality"], y_pred, average="macro")

    print("\n✅ F1-macro (teste):", round(f1, 4))
    print(classification_report(test_df["quality"], y_pred))

    # =========================
    # META.JSON PADRONIZADO (STREAMLIT ÚNICO)
    # =========================
    meta = {
        "model_kind": "autogluon",
        "artifact_path": "autogluon_model",
        "target": "quality",
        "features": df.drop(columns=["quality"]).columns.tolist(),
        "classes": sorted(df["quality"].unique().tolist()),
        "framework": "autogluon",
        "time_limit_sec": TIME_LIMIT,
        "macro_f1_test": float(f1)
    }

    with open(os.path.join(BASE_DIR, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print("\n✅ AutoGluon FINALIZADO")
    print(" - wine_autogluon/autogluon_model/")
    print(" - wine_autogluon/meta.json")

if __name__ == "__main__":
    main()


✅ AutoGluon salvo em: D:\TreinaRecife\Python do Zero até a Análise de Dados\aprendizado\Códigos\wine_autogluon\autogluon_model
[1/5] Carregando dados...
[2/5] EDA...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:00<00:00, 11895.92it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.11
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
Pytorch Version:    2.6.0+cu124
CUDA Version:       12.4
GPU Memory:         GPU 0: 6.00/6.00 GB
Total GPU Memory:   Free: 6.00 GB, Allocated: 0.00 GB, Total: 6.00 GB
GPU Count:          1
Memory Avail:       1.04 GB / 15.82 GB (6.6%)
Disk Space Avail:   884.49 GB / 1861.65 GB (47.5%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or 

[3/5] Feature Engineering...
[4/5] Treinando AutoGluon...


Beginning AutoGluon training ... Time limit = 45s
AutoGluon will save models to "D:\TreinaRecife\Python do Zero até a Análise de Dados\aprendizado\Códigos\wine_autogluon\autogluon_model\ds_sub_fit\sub_fit_ho"
Train Data Rows:    812
Train Data Columns: 15
Label Column:       quality
Problem Type:       multiclass
Preprocessing data ...
Fraction of data from classes with at least 10 examples that will be kept for training models: 0.9950738916256158
Train Data Class Count: 5
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    1189.03 MB
	Train Data (Original)  Memory Usage: 0.09 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator


✅ F1-macro (teste): 0.3104
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         7
           5       0.70      0.76      0.73        97
           6       0.61      0.62      0.62        92
           7       0.52      0.52      0.52        29
           8       0.00      0.00      0.00         3

    accuracy                           0.64       229
   macro avg       0.30      0.32      0.31       229
weighted avg       0.61      0.64      0.62       229


✅ AutoGluon FINALIZADO
 - wine_autogluon/autogluon_model/
 - wine_autogluon/meta.json
